# Credit Card Approval Prediction System
### Full EDA -> Preprocessing -> Feature Engineering -> Model Building -> Evaluation

This notebook mirrors the pipeline used in `train_pipeline.py`. It uses the same
`application_record.csv` / `credit_record.csv` schema as the well-known Kaggle
"Credit Card Approval Prediction" dataset. If you have the real dataset, drop the
two CSVs into the `data/` folder to replace the synthetic ones generated by
`data/generate_data.py`.

In [ ]:
# 1. Environment Setup & Package Installation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sns.set_style("darkgrid")
%matplotlib inline

## 2. Dataset Collection & Understanding

In [ ]:
app_df = pd.read_csv("data/application_record.csv")
credit_df = pd.read_csv("data/credit_record.csv")

print(app_df.shape, credit_df.shape)
app_df.head()

In [ ]:
app_df.info()
app_df.describe(include="all").T

## Feature Engineering: building the TARGET label

The real dataset has no explicit approve/reject column, it has a monthly
`STATUS` history per applicant. We convert this into a binary label:

- `STATUS` in `{2,3,4,5}` (60+ days past due through bad debt) -> applicant is
  **high risk** -> `TARGET = 1` (would be rejected)
- Otherwise (`0`, `1`, `C` paid off, `X` no loan that month) -> `TARGET = 0`
  (acceptable / approve)

In [ ]:
credit_df["IS_RISK"] = credit_df["STATUS"].isin(["2","3","4","5"]).astype(int)
risk_by_id = credit_df.groupby("ID")["IS_RISK"].max().reset_index()
risk_by_id.rename(columns={"IS_RISK": "TARGET"}, inplace=True)

df = app_df.merge(risk_by_id, on="ID", how="inner")
print(df.shape)
df["TARGET"].value_counts(normalize=True)

## 3. Data Visualization & Analysis (EDA)

In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(x="TARGET", data=df)
plt.title("Target Distribution (0=Approve, 1=Reject)")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df["AMT_INCOME_TOTAL"], bins=40, kde=True)
plt.title("Annual Income Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(y="NAME_INCOME_TYPE", hue="TARGET", data=df)
plt.title("Income Type vs Approval Outcome")
plt.show()

## 4. Data Preprocessing & Feature Engineering

In [ ]:
df.drop_duplicates(inplace=True)
df["OCCUPATION_TYPE"] = df["OCCUPATION_TYPE"].fillna("Unknown")

df["AGE_YEARS"] = (-df["DAYS_BIRTH"] // 365).astype(int)
df["YEARS_EMPLOYED"] = np.where(df["DAYS_EMPLOYED"] > 0, 0, (-df["DAYS_EMPLOYED"] // 365)).astype(int)

numeric_cols = ["CNT_CHILDREN","AMT_INCOME_TOTAL","AGE_YEARS","YEARS_EMPLOYED",
                 "CNT_FAM_MEMBERS","FLAG_WORK_PHONE","FLAG_PHONE","FLAG_EMAIL"]
categorical_cols = ["CODE_GENDER","FLAG_OWN_CAR","FLAG_OWN_REALTY","NAME_INCOME_TYPE",
                    "NAME_EDUCATION_TYPE","NAME_FAMILY_STATUS","NAME_HOUSING_TYPE","OCCUPATION_TYPE"]

model_df = df[numeric_cols + categorical_cols + ["TARGET"]].copy()

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    model_df[col] = le.fit_transform(model_df[col].astype(str))
    encoders[col] = le

model_df.head()

In [ ]:
X = model_df.drop(columns=["TARGET"])
y = model_df["TARGET"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

X_train.shape, X_test.shape

## 5. Machine Learning Model Building

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=10, class_weight="balanced", random_state=42),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.08, eval_metric="logloss", random_state=42,
                              scale_pos_weight=(y_train==0).sum()/max((y_train==1).sum(),1)),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    results[name] = {"model": model, "accuracy": acc}
    print(f"--- {name} --- accuracy={acc:.4f}")
    print(confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds))

In [ ]:
best_name = max(results, key=lambda k: results[k]["accuracy"])
best_model = results[best_name]["model"]
print("Best model:", best_name, "accuracy:", results[best_name]["accuracy"])

acc_df = pd.DataFrame({"model": list(results.keys()),
                        "accuracy": [results[n]["accuracy"] for n in results]})
plt.figure(figsize=(6,4))
sns.barplot(data=acc_df, x="accuracy", y="model")
plt.xlim(0,1)
plt.title("Model Comparison")
plt.show()

## 6. Save the best model

These artifacts are what `app.py` (the Flask application) loads at startup.

In [ ]:
import pickle, json, os

os.makedirs("model", exist_ok=True)
with open("model/best_model.pkl", "wb") as f: pickle.dump(best_model, f)
with open("model/scaler.pkl", "wb") as f: pickle.dump(scaler, f)
with open("model/encoders.pkl", "wb") as f: pickle.dump(encoders, f)
with open("model/feature_columns.json", "w") as f: json.dump(list(X.columns), f)
print("Saved.")